# 08 · Entity extraction readiness
Start with **skills/tools, titles and qualifications**, then add role-linked dates, certifications, projects and supported domain themes. Preserve evidence for every result. This preliminary notebook checks local files and demonstrates an existing helper on synthetic text; it does not train or measure extraction accuracy.

See the [execution plan](../plan/2026-09-13-entity-extraction/README.md), [data catalog](../docs/data/data-catalog.md), and notebooks 04/06. Production implementation remains proposed pending approval. All outputs contain aggregates or synthetic examples, capped at five records or 2,000 characters per cell.

In [ ]:
import json
import pandas as pd
import pyarrow.parquet as pq
from candidate_screener.config import RAW, INTERIM,  VOCAB, PROFILE_METRICS
from candidate_screener.data.sources import SOURCES
from candidate_screener.baselines.skills import extract_skills
from candidate_screener.annotation.redact import redact_text, assert_clean
print('Imports ready; no downloads or training.')

Imports ready; no downloads or training.


## 1. Local prerequisites
Presence is not provenance verification or repair readiness. DataTurks is resume entity supervision; SkillSpan is JD skill/knowledge supervision. ESCO and data-jobs support vocabulary, not gold spans. Missing NER data skips supervised exploration gracefully.

In [ ]:
readiness = []
for key in ('dataturks', 'skillspan', 'esco', 'data-jobs'):
    source = SOURCES[key]
    counts = [len(list((RAW / source.directory).glob(a.glob))) for a in source.artifacts]
    complete = all(count >= a.min_files for count, a in zip(counts, source.artifacts))
    readiness.append({'source': key, 'files': sum(counts), 'all_groups_present': complete})
print(pd.DataFrame(readiness).to_string(index=False))

   source  files  all_groups_present
dataturks      0               False
skillspan      0               False
     esco      0               False
data-jobs      0               False


In [ ]:
print(json.dumps({
    'interim_files': sum(p.is_file() for p in INTERIM.rglob('*')),
    'vocabulary_files': sum(p.is_file() for p in VOCAB.rglob('*')),
    'NER_status': ('Files present: verify provenance and repair readiness.'
                   if all(r['all_groups_present'] for r in readiness[:2])
                   else 'Prerequisites missing: supervised inspection/evaluation skipped.')
}, indent=2))

{
  "interim_files": 0,
  "vocabulary_files": 0,
  "NER_status": "Prerequisites missing: supervised inspection/evaluation skipped."
}


## 2. Historical profile, separate from local availability
These committed measurements explain the repair and vocabulary work. They are not observations from this notebook and do not imply that the measured files are present locally.

In [ ]:
profile = json.loads(PROFILE_METRICS.read_text())
checks = profile['checks']
print(json.dumps({
    'profile_generated': profile['generated'],
    'DataTurks': {k: checks['dataturks'][k] for k in ('docs', 'spans', 'broken_offsets', 'broken_offsets_pct')},
    'SkillSpan': {k: checks['skillspan'][k] for k in ('total_skill_spans', 'total_knowledge_spans')},
    'ESCO': {k: checks['esco'][k] for k in ('skills', 'alt_labels')},
    'data_jobs_rows': checks['data-jobs']['rows']
}, indent=2))

{
  "profile_generated": "2026-08-19T11:50:52+00:00",
  "DataTurks": {
    "docs": 220,
    "spans": 3556,
    "broken_offsets": 222,
    "broken_offsets_pct": 6.2
  },
  "SkillSpan": {
    "total_skill_spans": 4381,
    "total_knowledge_spans": 5236
  },
  "ESCO": {
    "skills": 13960,
    "alt_labels": 86694
  },
  "data_jobs_rows": 785741
}


## 3. Local parquet metadata only
No real document values are read. A1 fit labels are not NER gold; A13 fails. Djinni is unlabelled for NER. Respect the committed train/evaluation partition before developing rules on real examples. Raw row counts are not distinct resumes or cleaned evaluation sizes.

In [ ]:
local_parquets = []
for key in ('fit', 'djinni-cv', 'djinni-jd'):
    for path in sorted((RAW / SOURCES[key].directory).glob('*.parquet')):
        parquet = pq.ParquetFile(path)
        local_parquets.append({'source': key, 'file': path.name, 'rows': parquet.metadata.num_rows, 'columns': len(parquet.schema_arrow)})
print(pd.DataFrame(local_parquets).head(5).to_string(index=False) if local_parquets else 'No matching parquet files; inspection skipped.')

   source                       file   rows  columns
      fit  default-test-0000.parquet   1759        3
      fit default-train-0000.parquet   6241        3
djinni-cv default-train-0000.parquet 210250       11
djinni-jd default-train-0000.parquet 141897       10


In [ ]:
schemas = []
for key in ('fit', 'djinni-cv', 'djinni-jd'):
    paths = sorted((RAW / SOURCES[key].directory).glob('*.parquet'))
    if paths:
        schema = pq.ParquetFile(paths[0]).schema_arrow
        schemas.append({'source': key, 'first_columns': [f'{f.name}: {f.type}' for f in list(schema)[:5]]})
print(json.dumps(schemas, indent=2)[:2000])

[
  {
    "source": "fit",
    "first_columns": [
      "resume_text: string",
      "job_description_text: string",
      "label: string"
    ]
  },
  {
    "source": "djinni-cv",
    "first_columns": [
      "Position: string",
      "Moreinfo: string",
      "Looking For: string",
      "Highlights: string",
      "Primary Keyword: string"
    ]
  },
  {
    "source": "djinni-jd",
    "first_columns": [
      "Position: string",
      "Long Description: string",
      "Company Name: string",
      "Exp Years: string",
      "Primary Keyword: string"
    ]
  }
]


## 4. Synthetic keyword demonstration
The frozen matching helper uses a small handwritten list and repeats each match twice for term-frequency boosting. Deduplicate for this illustrative inventory. It supplies no offsets, canonical IDs, proficiency or calibrated confidence.

Redaction occurs inside this notebook. The redactor does not remove every name and changes offsets; real evidence needs equal-length masking or an explicit offset map. Commit aggregates and synthetic examples only.

In [ ]:
synthetic = {
    'resume': 'Synthetic candidate. Data Engineer. Built Python and SQL pipelines on AWS. BSc Computer Science. Contact demo@example.com. Learning Kubernetes.',
    'jd': 'Synthetic vacancy: Data Engineer. Python and SQL required; Docker preferred.'
}
safe_documents = {kind: redact_text(text) for kind, text in synthetic.items()}
assert_clean(pd.Series(safe_documents), 'synthetic notebook example')
print(json.dumps(safe_documents, indent=2))

{
  "resume": "Synthetic candidate. Data Engineer. Built Python and SQL pipelines on AWS. BSc Computer Science. Contact [EMAIL]. Learning Kubernetes.",
  "jd": "Synthetic vacancy: Data Engineer. Python and SQL required; Docker preferred."
}


In [ ]:
mentions = {kind: sorted(set(extract_skills(text).split())) for kind, text in safe_documents.items()}
print(json.dumps({
    'resume_mentions': mentions['resume'],
    'jd_mentions': mentions['jd'],
    'shared_mentions': sorted(set(mentions['resume']) & set(mentions['jd'])),
    'jd_mentions_not_evidenced_in_resume': sorted(set(mentions['jd']) - set(mentions['resume']))
}, indent=2))

{
  "resume_mentions": [
    "aws",
    "kubernetes",
    "python",
    "sql"
  ],
  "jd_mentions": [
    "docker",
    "python",
    "sql"
  ],
  "shared_mentions": [
    "python",
    "sql"
  ],
  "jd_mentions_not_evidenced_in_resume": [
    "docker"
  ]
}


Not evidenced does not mean the candidate lacks a skill. Learning Kubernetes counts as a mention, not applied proficiency. Required/preferred status must come from the JD itself, not keyword overlap or an ESCO occupation relation.

In [ ]:
cases = ['Built C++ services.', 'Ready to go to the next project.', 'No Python experience; interested in learning Python.']
print(json.dumps([{'synthetic_text': text, 'helper_mentions': sorted(set(extract_skills(text).split()))} for text in cases], indent=2))

[
  {
    "synthetic_text": "Built C++ services.",
    "helper_mentions": []
  },
  {
    "synthetic_text": "Ready to go to the next project.",
    "helper_mentions": [
      "go"
    ]
  },
  {
    "synthetic_text": "No Python experience; interested in learning Python.",
    "helper_mentions": [
      "python"
    ]
  }
]


The trailing word boundary misses ordinary **C++** mentions; **go** can be an English verb; negation and aspiration remain positive matches. Keep these as future extractor requirements. Synthetic examples cannot establish gold precision/recall/F1.

## 5. Evidence schema and useful information

| Priority | Information | Evidence / interpretation |
|---|---|---|
| First | Skills, tools, titles, degrees and fields | Exact mention, context, optional canonical mapping |
| Next | Certifications, employment dates | Role association; stated versus computed duration; overlapping roles |
| Next | Projects, responsibilities, outcomes | Activity, result, unit and context |
| Later | Explicit domain themes | Derived summaries linked to supporting spans |

An entity record carries `doc_id`, `source`, `text_version`, `start`, `end`, `label`, `surface`, nullable `canonical_id`, `method`, `extractor_version`, and nullable `confidence`. Half-open coordinates satisfy `text[start:end] == surface`. Relations need entity IDs to associate skills/dates with roles. Preserve immutable source text and version any normalisation. Rules should not invent probability scores.

Employers and institutions are context, not prestige scores. Names, contacts, age, gender, nationality and inferred protected traits are not screening features. Themes describe documented work, not inferred personality. Certifications, activities and themes lack dedicated initial gold supervision.

## 6. Execution steps

1. Finish readiness, resolve [EE1–EE5](../plan/2026-09-13-entity-extraction/README.md#open-questions), and approve implementation scope.
2. Restore and verify B1/B2/D2 and ESCO via registered fetch/manual instructions. C1/C2 additionally require contamination exclusion; C3 is unsuitable for offsets.
3. Repair B1 inclusive offsets, quarantine ambiguous repairs, and freeze document-group evaluation. Original Skills blobs stay gold; automatic segmentation is silver. Retain B2 splits and independent BIO layers.
4. Build classical phrase/rule extraction in the package with versioned aliases, punctuation and context handling. Keep notebooks thin and the matching helper unchanged.
5. Evaluate exact span-and-label precision/recall/F1, per-class support and B1 CV mean/std; report B2 per layer and split. Evaluate normalisation separately; A2 coverage is descriptive, not accuracy.
6. Compare BERT NER after the baseline under the same protocol and agreed compute budget, handling subword alignment and long documents.
7. Integrate evidence and themes after validation. Test ranking separately on frozen pools and each in-domain stratum; Recall@10 remains headline and Precision@k carries its soft ceiling. Extra annotation requires re-budgeting the one-session cap.

This notebook changes no model, label, source file or frozen benchmark record.